# HNSW Hybrid Retrieval — TF-IDF + Sentence Embeddings

This notebook builds **two HNSW indexes** and combines them at query time.

| Index | Embedding | Strengths |
|-------|-----------|----------|
| Sparse | TF-IDF (unigrams + bigrams) | Exact keyword match: "Roman", "Baroque", country names |
| Dense  | `all-MiniLM-L6-v2` sentence embeddings | Semantic match: "beach holiday" ≈ "coastal turquoise coves" |
| **Hybrid** | Both fused | Best of both worlds |

### Fusion strategies covered
1. **Reciprocal Rank Fusion (RRF)** — rank-based, no score normalisation needed
2. **Linear score fusion** — normalise each system's scores to [0,1], then blend with weight `α`

### Notebook structure
| Step | What happens |
|------|--------------|
| 1 | Data (46 Europe places) |
| 2 | Sparse embeddings — TF-IDF |
| 3 | Dense embeddings — sentence-transformers |
| 4 | HNSW class |
| 5 | Build both indexes |
| 6 | Hybrid search — RRF |
| 7 | Hybrid search — linear score fusion |
| 8 | Side-by-side comparison: sparse vs dense vs hybrid |
| 9 | Recall evaluation — all three systems |
| 10 | Incremental insertion into both indexes |

## 0. Install dependencies

In [7]:
%pip install numpy scikit-learn sentence-transformers 

Note: you may need to restart the kernel to use updated packages.


## 1. Imports

In [8]:
import math
import random
import heapq
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

## 2. Data — 46 Europe travel places

In [9]:
PLACES = [
    # ── France ──
    {"name": "Eiffel Tower", "country": "France",
     "description": "Iconic iron lattice tower on the Champ de Mars in Paris. Symbol of romance and French engineering. Stunning views of the city skyline.",
     "best_time": "April–June", "category": "landmark"},
    {"name": "Mont Saint-Michel", "country": "France",
     "description": "Medieval abbey perched on a tidal island in Normandy. Surrounded by vast sandflats and tidal waters. One of France's most visited monuments.",
     "best_time": "May–September", "category": "heritage"},
    {"name": "Palace of Versailles", "country": "France",
     "description": "Opulent royal château with baroque gardens outside Paris. Former residence of Louis XIV. UNESCO World Heritage Site.",
     "best_time": "April–October", "category": "heritage"},
    {"name": "French Riviera", "country": "France",
     "description": "Mediterranean coastline with glamorous beach resorts, Nice, Cannes and Monaco. Crystal blue waters and luxury yachts. Vibrant nightlife.",
     "best_time": "June–August", "category": "beach"},
    {"name": "Loire Valley", "country": "France",
     "description": "Valley of châteaux along the Loire river. Renaissance architecture, vineyards and cycling paths through lush French countryside.",
     "best_time": "May–October", "category": "nature"},
    # ── Italy ──
    {"name": "Colosseum", "country": "Italy",
     "description": "Ancient Roman amphitheatre in the heart of Rome. Largest amphitheatre ever built. Iconic symbol of Imperial Rome and gladiatorial combat.",
     "best_time": "March–May", "category": "heritage"},
    {"name": "Amalfi Coast", "country": "Italy",
     "description": "Dramatic cliffside villages along the Tyrrhenian Sea south of Naples. Pastel-coloured houses, limoncello, and turquoise coves.",
     "best_time": "May–October", "category": "beach"},
    {"name": "Venice Canals", "country": "Italy",
     "description": "City built on water with gondolas navigating narrow canals. St Mark's Basilica and Doge's Palace reflect Byzantine grandeur. Carnival in February.",
     "best_time": "April–June", "category": "landmark"},
    {"name": "Tuscany Hills", "country": "Italy",
     "description": "Rolling hills with cypress trees, vineyards and medieval hilltop towns. Chianti wine country. Siena and San Gimignano nearby.",
     "best_time": "April–October", "category": "nature"},
    {"name": "Pompeii", "country": "Italy",
     "description": "Ancient Roman city preserved under volcanic ash from Mount Vesuvius eruption in AD79. Extraordinary window into Roman daily life.",
     "best_time": "March–May", "category": "heritage"},
    {"name": "Cinque Terre", "country": "Italy",
     "description": "Five colourful fishing villages clinging to cliffs on the Ligurian coast. Hiking trails with dramatic sea views. Fresh seafood and pesto.",
     "best_time": "May–September", "category": "beach"},
    {"name": "Dolomites", "country": "Italy",
     "description": "Alpine mountain range in northeastern Italy with dramatic jagged peaks. World-class skiing in winter and hiking in summer. UNESCO heritage.",
     "best_time": "June–September", "category": "nature"},
    # ── Spain ──
    {"name": "Sagrada Familia", "country": "Spain",
     "description": "Gaudí's unfinished basilica in Barcelona. Extraordinary organic architecture blending Gothic and Art Nouveau. Under construction since 1882.",
     "best_time": "March–May", "category": "landmark"},
    {"name": "Alhambra", "country": "Spain",
     "description": "Moorish palace complex in Granada with intricate Islamic architecture. Stunning views over the Sierra Nevada. UNESCO World Heritage Site.",
     "best_time": "March–June", "category": "heritage"},
    {"name": "Ibiza", "country": "Spain",
     "description": "Island in the Balearics famous for nightlife and electronic music. Also has quiet coves, old town Dalt Vila and crystal-clear Mediterranean waters.",
     "best_time": "June–September", "category": "beach"},
    {"name": "Park Güell", "country": "Spain",
     "description": "Gaudí's mosaic-covered public park on a hill above Barcelona. Colourful tiled benches and organic structures with sweeping city views.",
     "best_time": "April–October", "category": "landmark"},
    {"name": "Camino de Santiago", "country": "Spain",
     "description": "Ancient pilgrimage route across northern Spain ending at Santiago Cathedral. 800km trail through diverse landscapes, villages and vineyards.",
     "best_time": "May–September", "category": "nature"},
    # ── Greece ──
    {"name": "Santorini", "country": "Greece",
     "description": "Volcanic island with iconic white-washed buildings and blue-domed churches. Dramatic caldera views and stunning sunsets from Oia.",
     "best_time": "June–September", "category": "beach"},
    {"name": "Acropolis of Athens", "country": "Greece",
     "description": "Ancient citadel above Athens containing the Parthenon temple. Masterpiece of ancient Greek architecture. UNESCO World Heritage Site.",
     "best_time": "April–May", "category": "heritage"},
    {"name": "Meteora", "country": "Greece",
     "description": "Eastern Orthodox monasteries perched on natural sandstone pillars in central Greece. Otherworldly landscape with dramatic rock formations.",
     "best_time": "April–June", "category": "heritage"},
    {"name": "Mykonos", "country": "Greece",
     "description": "Cyclades island renowned for cosmopolitan beach clubs, windmills and vibrant LGBTQ+ nightlife. Picturesque Little Venice waterfront.",
     "best_time": "June–September", "category": "beach"},
    {"name": "Crete", "country": "Greece",
     "description": "Largest Greek island with Minoan ruins at Knossos, Samaria Gorge hikes and diverse beaches from pink sand to palm-lined coves.",
     "best_time": "May–October", "category": "nature"},
    # ── Germany ──
    {"name": "Neuschwanstein Castle", "country": "Germany",
     "description": "Fairytale castle commissioned by King Ludwig II in the Bavarian Alps. Inspiration for Disney's Sleeping Beauty castle. Stunning Alpine backdrop.",
     "best_time": "May–September", "category": "landmark"},
    {"name": "Berlin Wall Memorial", "country": "Germany",
     "description": "Memorial site along the former Berlin Wall with preserved segments and documentation centre. Powerful reminder of Cold War division.",
     "best_time": "Year-round", "category": "heritage"},
    {"name": "Black Forest", "country": "Germany",
     "description": "Dense forested mountain range in southwest Germany. Cuckoo clocks, kirsch cake, thermal baths and hiking through dense dark spruce woods.",
     "best_time": "May–October", "category": "nature"},
    {"name": "Rhine Valley", "country": "Germany",
     "description": "Romantic Rhine Gorge with medieval castles on hillsides above the river. Wine villages and boat cruises through UNESCO protected landscape.",
     "best_time": "May–October", "category": "nature"},
    # ── Austria ──
    {"name": "Vienna Schönbrunn Palace", "country": "Austria",
     "description": "Baroque imperial palace and gardens of the Habsburg dynasty in Vienna. 1441 rooms and a hilltop Gloriette with panoramic city views.",
     "best_time": "April–October", "category": "heritage"},
    {"name": "Hallstatt", "country": "Austria",
     "description": "Picturesque lakeside village in the Salzkammergut mountains. Steep salt mine history, wooden architecture and mirror-like Alpine lake.",
     "best_time": "May–September", "category": "nature"},
    # ── Switzerland ──
    {"name": "Matterhorn", "country": "Switzerland",
     "description": "Iconic pyramid-shaped peak on the Swiss-Italian border near Zermatt. One of the highest peaks in the Alps. World-class skiing resort.",
     "best_time": "June–September", "category": "nature"},
    {"name": "Lake Geneva", "country": "Switzerland",
     "description": "Large crescent-shaped lake on the Swiss-French border. Vineyards on terrace hillsides, medieval Chillon Castle and the Jet d'Eau fountain.",
     "best_time": "May–September", "category": "nature"},
    # ── Portugal ──
    {"name": "Lisbon Historic Trams", "country": "Portugal",
     "description": "Iconic yellow trams winding through steep Alfama neighbourhood cobblestone streets. Fado music, pastéis de nata and Moorish castle views.",
     "best_time": "March–May", "category": "landmark"},
    {"name": "Algarve Beaches", "country": "Portugal",
     "description": "Atlantic coastline with golden limestone cliffs, sea caves and turquoise lagoons. Surfing, seafood restaurants and historic fishing villages.",
     "best_time": "June–September", "category": "beach"},
    {"name": "Sintra Palaces", "country": "Portugal",
     "description": "Fairytale town in forested hills near Lisbon with Romantic-era palaces including Pena Palace and the Moorish Castle ruins.",
     "best_time": "March–October", "category": "heritage"},
    # ── Czech Republic ──
    {"name": "Prague Old Town", "country": "Czech Republic",
     "description": "Medieval old town with astronomical clock, Gothic churches and Baroque palaces around cobblestone squares. Bohemian beer culture.",
     "best_time": "April–October", "category": "heritage"},
    {"name": "Český Krumlov", "country": "Czech Republic",
     "description": "Tiny UNESCO heritage town in Bohemia with a massive castle above a river bend. Baroque theatre, rafting and atmospheric cobblestone lanes.",
     "best_time": "May–September", "category": "heritage"},
    # ── Netherlands ──
    {"name": "Keukenhof Gardens", "country": "Netherlands",
     "description": "World's largest flower garden near Amsterdam with 7 million tulip bulbs in bloom. Open only in spring. Windmill and Dutch pastoral landscape.",
     "best_time": "March–May", "category": "nature"},
    {"name": "Amsterdam Canals", "country": "Netherlands",
     "description": "17th-century canal ring with narrow gabled houses, houseboats and cycling culture. Van Gogh Museum, Rijksmuseum and Anne Frank House nearby.",
     "best_time": "April–September", "category": "landmark"},
    # ── Scandinavia ──
    {"name": "Norwegian Fjords", "country": "Norway",
     "description": "Spectacular glacially carved inlets with waterfalls dropping from steep cliffs. Cruise or kayak through Geirangerfjord and Nærøyfjord UNESCO sites.",
     "best_time": "May–September", "category": "nature"},
    {"name": "Northern Lights Iceland", "country": "Iceland",
     "description": "Aurora borealis dancing across Arctic skies. Hot springs, geysers, black sand beaches and midnight sun in the land of fire and ice.",
     "best_time": "September–March", "category": "nature"},
    {"name": "Stockholm Gamla Stan", "country": "Sweden",
     "description": "Stockholm's medieval old town on a small island. Colourful 17th-century buildings, Royal Palace, Nobel Museum and winding cobblestone alleys.",
     "best_time": "June–August", "category": "heritage"},
    # ── Eastern Europe ──
    {"name": "Budapest Thermal Baths", "country": "Hungary",
     "description": "City built on natural hot springs with grand Ottoman and Art Nouveau bath houses. Széchenyi and Gellért baths are the most famous.",
     "best_time": "Year-round", "category": "landmark"},
    {"name": "Dubrovnik Old City", "country": "Croatia",
     "description": "Walled medieval city on the Adriatic coast. Walk the city walls above terracotta rooftops and turquoise sea. Game of Thrones filming location.",
     "best_time": "May–June", "category": "heritage"},
    {"name": "Plitvice Lakes", "country": "Croatia",
     "description": "Cascading turquoise lakes and waterfalls in a forested canyon. UNESCO national park with wooden boardwalks above crystal-clear mineral waters.",
     "best_time": "April–October", "category": "nature"},
    {"name": "Transylvania", "country": "Romania",
     "description": "Region of medieval castles, fortified churches and Carpathian mountain forests. Bran Castle, Dracula legend and authentic rural village life.",
     "best_time": "May–September", "category": "heritage"},
    # ── UK & Ireland ──
    {"name": "Scottish Highlands", "country": "Scotland",
     "description": "Wild landscape of glens, lochs and dramatic mountains. Loch Ness, Glencoe valley, Eilean Donan castle and whisky distillery tours.",
     "best_time": "May–September", "category": "nature"},
    {"name": "Cliffs of Moher", "country": "Ireland",
     "description": "Dramatic 214-metre sea cliffs on the Atlantic coast of County Clare. Puffin colonies, crashing waves and clear days with Aran Islands views.",
     "best_time": "April–September", "category": "nature"},
    {"name": "Stonehenge", "country": "England",
     "description": "Prehistoric stone circle monument in Wiltshire. Built 3000–1500 BC. Alignment with summer solstice sunrise. UNESCO World Heritage Site.",
     "best_time": "April–September", "category": "heritage"},
]

# Build one combined text doc per place — used by both embedding types
DOCS = [
    f"{p['name']} {p['country']} {p['category']} {p['description']}"
    for p in PLACES
]

N = len(PLACES)
print(f"Loaded {N} places. Example doc:\n  {DOCS[0]}")

Loaded 47 places. Example doc:
  Eiffel Tower France landmark Iconic iron lattice tower on the Champ de Mars in Paris. Symbol of romance and French engineering. Stunning views of the city skyline.


## 3. Sparse embeddings — TF-IDF

Fit on the dataset, L2-normalise so dot product = cosine similarity.

In [10]:
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
tfidf_matrix     = tfidf_vectorizer.fit_transform(DOCS).toarray()
sparse_embs      = normalize(tfidf_matrix, norm='l2')   # shape: (N, vocab_size)

print(f"Sparse embeddings : {sparse_embs.shape}  (N × vocab)")
print(f"Vocabulary size   : {len(tfidf_vectorizer.vocabulary_)} terms")
print(f"Unit-norm check   : {np.linalg.norm(sparse_embs[0]):.6f}")

Sparse embeddings : (47, 1548)  (N × vocab)
Vocabulary size   : 1548 terms
Unit-norm check   : 1.000000


## 4. Dense embeddings — sentence-transformers

`all-MiniLM-L6-v2` is a fast 384-dimensional model trained on 1B+ sentence pairs.
It understands paraphrases and semantic intent — things TF-IDF misses completely.

Examples of what dense embeddings handle that TF-IDF cannot:
- Query `"summer swim spot"` → matches `"turquoise coves"` and `"Mediterranean beach"`
- Query `"fairy tale architecture"` → matches both `"Gothic"` and `"Moorish"`
- Query `"cold weather wonder"` → matches `"aurora borealis"` and `"Arctic skies"`

In [11]:
# Download model on first run (~90MB), cached locally afterwards
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

# encode() returns L2-normalised vectors by default when normalize_embeddings=True
dense_embs = sentence_model.encode(
    DOCS[:2],
    normalize_embeddings=True,   # unit vectors → dot product = cosine similarity
    show_progress_bar=True,
    batch_size=32,
)

print(f"\nDense embeddings  : {dense_embs.shape}  (N × 384)")
print(f"Unit-norm check   : {np.linalg.norm(dense_embs[0]):.6f}")

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.67it/s]


Dense embeddings  : (2, 384)  (N × 384)
Unit-norm check   : 1.000000


In [12]:
# encode() returns L2-normalised vectors by default when normalize_embeddings=True
dense_embs = sentence_model.encode(
    DOCS,
    normalize_embeddings=True,   # unit vectors → dot product = cosine similarity
    show_progress_bar=True,
    batch_size=32,
)

print(f"\nDense embeddings  : {dense_embs.shape}  (N × 384)")
print(f"Unit-norm check   : {np.linalg.norm(dense_embs[0]):.6f}")

Batches: 100%|██████████| 2/2 [00:00<00:00, 10.25it/s]


Dense embeddings  : (47, 384)  (N × 384)
Unit-norm check   : 1.000000


In [13]:
len(dense_embs)

47

## 5. HNSW class

Works for any unit-normalised vector regardless of dimension.
Instantiated twice below: once for sparse, once for dense.

In [14]:
class HNSWIndex:
    def __init__(self, M=16, M0=None, ef_construction=200, ef_search=50):
        self.M               = M
        self.M0              = M0 if M0 is not None else 2 * M
        self.ef_construction = ef_construction
        self.ef_search       = ef_search
        self.m_L             = 1.0 / math.log(M) if M > 1 else 1.0
        self.vectors         = []
        self.labels          = []
        self.max_layer       = -1
        self.entry_point     = None
        self.graphs          = defaultdict(lambda: defaultdict(set))

    def _sim(self, a, b):
        # Dot product of two unit vectors == cosine similarity
        return float(np.dot(a, b))

    def _random_level(self):
        # Exponential distribution → ~63% nodes stay at layer 0
        return int(-math.log(random.random()) * self.m_L)

    def _search_layer(self, query, entry_ids, ef, layer):
        visited    = set(entry_ids)
        candidates = []
        found      = []
        for eid in entry_ids:
            s = self._sim(query, self.vectors[eid])
            heapq.heappush(candidates, (-s, eid))
            heapq.heappush(found,      (-s, eid))
        while candidates:
            neg_sim_c, c = heapq.heappop(candidates)
            if -neg_sim_c < -found[0][0]:   # best remaining < worst found → stop
                break
            for nb in self.graphs[layer][c]:
                if nb in visited:
                    continue
                visited.add(nb)
                s_n = self._sim(query, self.vectors[nb])
                if len(found) < ef or s_n > -found[0][0]:
                    heapq.heappush(candidates, (-s_n, nb))
                    heapq.heappush(found,      (-s_n, nb))
                    if len(found) > ef:
                        heapq.heappop(found)
        return [(-ns, nid) for ns, nid in found]

    def _select_neighbors(self, query, candidates, M):
        # Diversity heuristic: keep c only if it's closer to query than to any already-selected node
        selected = []
        for sim_c, c in sorted(candidates, key=lambda x: -x[0]):
            if len(selected) >= M:
                break
            if all(self._sim(self.vectors[c], self.vectors[s]) <= sim_c for _, s in selected):
                selected.append((sim_c, c))
        return selected

    def insert(self, vector, label=None):
        node_id   = len(self.vectors)
        self.vectors.append(vector)
        self.labels.append(label)
        new_level = self._random_level()
        entry_ids = [self.entry_point] if self.entry_point is not None else []
        if self.entry_point is not None:
            for layer in range(self.max_layer, new_level, -1):
                entry_ids = [self._search_layer(vector, entry_ids, 1, layer)[0][1]]
        for layer in range(min(new_level, self.max_layer), -1, -1):
            if not entry_ids:
                break
            candidates = self._search_layer(vector, entry_ids, self.ef_construction, layer)
            M_layer    = self.M0 if layer == 0 else self.M
            neighbors  = self._select_neighbors(vector, candidates, M_layer)
            for _, n_id in neighbors:
                self.graphs[layer][node_id].add(n_id)
                self.graphs[layer][n_id].add(node_id)
                if len(self.graphs[layer][n_id]) > M_layer:
                    cands = [(self._sim(self.vectors[n_id], self.vectors[nb]), nb)
                             for nb in self.graphs[layer][n_id]]
                    self.graphs[layer][n_id] = {nb for _, nb in self._select_neighbors(
                        self.vectors[n_id], cands, M_layer)}
            entry_ids = [n for _, n in neighbors]
        if new_level > self.max_layer:
            self.max_layer   = new_level
            self.entry_point = node_id
        return node_id

    def search(self, query, k=5):
        if self.entry_point is None:
            return []
        entry_ids = [self.entry_point]
        for layer in range(self.max_layer, 0, -1):
            entry_ids = [self._search_layer(query, entry_ids, 1, layer)[0][1]]
        results = self._search_layer(query, entry_ids, self.ef_search, 0)
        results.sort(key=lambda x: -x[0])
        return [(sim, self.labels[nid]) for sim, nid in results[:k]]


print("HNSWIndex class ready.")

HNSWIndex class ready.


## 6. Build both indexes

One index per embedding type, same HNSW hyperparameters.

In [15]:
HNSW_PARAMS = dict(M=8, M0=16, ef_construction=100, ef_search=30)

random.seed(42)
sparse_index = HNSWIndex(**HNSW_PARAMS)
for vec, place in zip(sparse_embs, PLACES):
    sparse_index.insert(vec, label=place)
print(f"Sparse index: {len(sparse_index.vectors)} nodes, top layer={sparse_index.max_layer}")

random.seed(42)
dense_index = HNSWIndex(**HNSW_PARAMS)
for vec, place in zip(dense_embs, PLACES):
    dense_index.insert(vec, label=place)
print(f"Dense index : {len(dense_index.vectors)} nodes, top layer={dense_index.max_layer}")

Sparse index: 47 nodes, top layer=2
Dense index : 47 nodes, top layer=2


## 7. Query embedding helpers

Both embedding functions take a plain text string and return a unit-normalised vector
in the same space as the corresponding index.

In [16]:
def embed_sparse(text: str) -> np.ndarray:
    """TF-IDF + L2-normalise. Uses the vocabulary fitted on DOCS."""
    vec  = tfidf_vectorizer.transform([text]).toarray()[0]
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec


def embed_dense(text: str) -> np.ndarray:
    """Sentence-transformer encode, already unit-normalised by the model."""
    return sentence_model.encode(text, normalize_embeddings=True)


print("Query embedders defined.")

Query embedders defined.


## 8. Hybrid retrieval — Reciprocal Rank Fusion (RRF)

RRF is the simplest and most robust fusion method.  
It ignores raw similarity scores entirely and works purely on **rank**.

$$\text{RRF}(d) = \sum_{i} \frac{1}{k + r_i(d)}$$

- $r_i(d)$ = rank of document $d$ in retriever $i$ (1 = top)
- $k$ = constant (default 60) that dampens the impact of very high ranks
- Summed over all retrievers (sparse + dense here)

**Why RRF is robust**: scores from different systems are on incomparable scales.
RRF sidesteps the normalisation problem completely.

In [17]:
def hybrid_rrf(query_text: str, k: int = 5, rrf_k: int = 60,
               fetch: int = 20) -> list:
    """
    Retrieve top-k results by combining sparse and dense HNSW indexes
    using Reciprocal Rank Fusion.

    Parameters
    ----------
    query_text : free-text query string
    k          : final number of results to return
    rrf_k      : RRF smoothing constant (60 is the standard default)
    fetch      : how many candidates to pull from each index before fusion
                 (should be >> k to give RRF enough candidates to rerank)
    """
    # --- Step 1: retrieve candidates from each index ---
    sparse_results = sparse_index.search(embed_sparse(query_text), k=fetch)
    dense_results  = dense_index.search(embed_dense(query_text),   k=fetch)

    # --- Step 2: assign RRF scores ---
    # Use place name as a stable ID for merging the two ranked lists
    rrf_scores: dict[str, float] = defaultdict(float)
    place_by_name: dict[str, dict] = {}

    for rank, (_, place) in enumerate(sparse_results, start=1):
        name = place['name']
        rrf_scores[name]  += 1.0 / (rrf_k + rank)
        place_by_name[name] = place

    for rank, (_, place) in enumerate(dense_results, start=1):
        name = place['name']
        rrf_scores[name]  += 1.0 / (rrf_k + rank)
        place_by_name[name] = place

    # --- Step 3: sort by fused score, return top-k ---
    ranked = sorted(rrf_scores.items(), key=lambda x: -x[1])
    return [(score, place_by_name[name]) for name, score in ranked[:k]]


print("hybrid_rrf() defined.")

hybrid_rrf() defined.


## 9. Hybrid retrieval — linear score fusion

An alternative to RRF that **preserves score magnitudes** after normalisation.

$$\text{score}(d) = \alpha \cdot \hat{s}_{\text{dense}}(d) + (1-\alpha) \cdot \hat{s}_{\text{sparse}}(d)$$

where $\hat{s}$ means min-max normalised to [0, 1] within each result set.

- $\alpha = 0.5$ weights both equally
- $\alpha \to 1.0$ makes it pure dense; $\alpha \to 0.0$ makes it pure sparse
- Tune $\alpha$ on a validation set for best recall on your data

In [18]:
def _minmax_normalise(scored_list: list) -> dict:
    """Normalise a list of (score, label) to a {name: normalised_score} dict."""
    if not scored_list:
        return {}
    scores = [s for s, _ in scored_list]
    lo, hi = min(scores), max(scores)
    span   = hi - lo if hi != lo else 1.0
    return {place['name']: (s - lo) / span for s, place in scored_list}


def hybrid_linear(query_text: str, k: int = 5, alpha: float = 0.5,
                  fetch: int = 20) -> list:
    """
    Retrieve top-k results by linearly blending normalised sparse and dense scores.

    Parameters
    ----------
    alpha : weight for dense score (1-alpha goes to sparse)
    fetch : candidates fetched from each index before fusion
    """
    sparse_results = sparse_index.search(embed_sparse(query_text), k=fetch)
    dense_results  = dense_index.search(embed_dense(query_text),   k=fetch)

    # Min-max normalise each list independently to [0, 1]
    sparse_norm = _minmax_normalise(sparse_results)
    dense_norm  = _minmax_normalise(dense_results)

    # Union of all candidate names
    all_names = set(sparse_norm) | set(dense_norm)

    # Store place metadata (prefer dense dict, fall back to sparse)
    place_by_name = {place['name']: place for _, place in sparse_results}
    place_by_name.update({place['name']: place for _, place in dense_results})

    # Blend: missing from one side → contributes 0 from that side
    blended = {
        name: alpha * dense_norm.get(name, 0.0) + (1 - alpha) * sparse_norm.get(name, 0.0)
        for name in all_names
    }

    ranked = sorted(blended.items(), key=lambda x: -x[1])
    return [(score, place_by_name[name]) for name, score in ranked[:k]]


print("hybrid_linear() defined.")

hybrid_linear() defined.


## 10. Side-by-side comparison

For each query, print results from all four retrievers:
- Sparse-only (TF-IDF HNSW)
- Dense-only (sentence-transformer HNSW)
- Hybrid RRF
- Hybrid linear (α = 0.5)

Notice how queries with **exact keywords** favour sparse, while **paraphrase/semantic** queries favour dense.

In [19]:
def print_comparison(query_text: str, k: int = 5) -> None:
    sparse_index.ef_search = 50
    dense_index.ef_search  = 50

    s_results  = sparse_index.search(embed_sparse(query_text), k=k)
    d_results  = dense_index.search(embed_dense(query_text),   k=k)
    rrf        = hybrid_rrf(query_text,    k=k, fetch=20)
    linear     = hybrid_linear(query_text, k=k, alpha=0.5, fetch=20)

    print(f"\n{'─'*90}")
    print(f"Query: '{query_text}'")
    print(f"{'─'*90}")
    header = f"{'#':<3}  {'SPARSE (TF-IDF)':<26}  {'DENSE (SentenceTransformer)':<26}  {'HYBRID RRF':<26}  {'HYBRID Linear α=0.5'}"
    print(header)
    print(f"{'─'*90}")
    for i in range(k):
        s = s_results[i][1]['name']  if i < len(s_results) else ""
        d = d_results[i][1]['name']  if i < len(d_results) else ""
        r = rrf[i][1]['name']        if i < len(rrf)        else ""
        l = linear[i][1]['name']     if i < len(linear)     else ""
        print(f"{i+1:<3}  {s:<26}  {d:<26}  {r:<26}  {l}")


QUERIES = [
    # Semantic — TF-IDF will struggle, dense should win
    "summer swim spot with crystal water",
    "fairy tale palace with ornate gardens",
    "cold weather natural wonder",
    "pilgrimage or spiritual journey on foot",
    # Keyword-heavy — TF-IDF should do well
    "ancient Roman amphitheatre",
    "UNESCO heritage medieval cobblestone",
    # Balanced — hybrid should outperform both
    "beach holiday cliffs Mediterranean",
    "Alpine mountains skiing hiking",
    "wine vineyard countryside cycling",
]

for q in QUERIES:
    print_comparison(q, k=5)


──────────────────────────────────────────────────────────────────────────────────────────
Query: 'summer swim spot with crystal water'
──────────────────────────────────────────────────────────────────────────────────────────
#    SPARSE (TF-IDF)             DENSE (SentenceTransformer)  HYBRID RRF                  HYBRID Linear α=0.5
──────────────────────────────────────────────────────────────────────────────────────────
1    Venice Canals               Plitvice Lakes              Meteora                     Plitvice Lakes
2    Meteora                     Hallstatt                   Keukenhof Gardens           Venice Canals
3    Sintra Palaces              Santorini                   Acropolis of Athens         Hallstatt
4    Rhine Valley                Stonehenge                  Stockholm Gamla Stan        Santorini
5    Park Güell                  Meteora                     Venice Canals               Stonehenge

─────────────────────────────────────────────────────────────────

## 10b. More query examples — travel intent categories

These queries cover common trip-planning intents. They stress-test the retrieval systems in different ways:

| Category | Why it's interesting |
|----------|---------------------|
| **Summer beaches** | Short, intent-heavy — dense wins on semantics, sparse may hit literal "beach" |
| **Romantic / couples** | No keyword in dataset — pure semantic task for dense |
| **Adventure & outdoors** | Overlaps with multiple places; hybrid should surface diverse set |
| **City breaks & culture** | Mix of landmark keywords and cultural context |
| **Family / kids** | No "kids" keyword in dataset — dense must infer from context |
| **Food & drink** | Requires semantic match to wine/cuisine descriptions |
| **Photography spots** | Entirely semantic — no "photography" in dataset |
| **Off the beaten path** | Vague intent — tests which system understands "lesser-known" |

In [23]:
MORE_QUERIES = {
    # ── Summer & beaches ──────────────────────────────────────────────────────
    "Summer beaches": [
        "summer beaches",                            # shortest possible; forces semantic inference
        "warm sandy beach with clear blue sea",
        "beach resort with turquoise water and sunshine",
        "Mediterranean beach holiday in summer",
    ],

    # ── Romantic / couples ────────────────────────────────────────────────────
    "Romantic / couples": [
        "romantic getaway for couples",
        "sunset views and candlelit atmosphere",
        "honeymoon destination with luxury and charm",
    ],

    # ── Adventure & outdoors ──────────────────────────────────────────────────
    "Adventure & outdoors": [
        "outdoor adventure with hiking and waterfalls",
        "extreme mountain trekking and skiing",
        "kayaking through dramatic fjords",
        "cliff walks and wild Atlantic scenery",
    ],

    # ── City breaks & culture ─────────────────────────────────────────────────
    "City breaks & culture": [
        "vibrant city break with art and nightlife",
        "historic old town with cobblestone streets and cafes",
        "world-class museums and modern architecture",
    ],

    # ── Family / kids ─────────────────────────────────────────────────────────
    "Family / kids": [
        "fun destination for families with children",
        "fairy tale castle kids will love",
        "safe beach with calm shallow water for kids",
    ],

    # ── Food & drink ──────────────────────────────────────────────────────────
    "Food & drink": [
        "wine tasting and vineyard tours",
        "fresh seafood by the sea",
        "local street food and traditional cuisine",
    ],

    # ── Photography spots ─────────────────────────────────────────────────────
    "Photography spots": [
        "stunning sunrise and sunset photography location",
        "colourful architecture perfect for photos",
        "dramatic landscape for landscape photography",
    ],

    # ── Off the beaten path ───────────────────────────────────────────────────
    "Off the beaten path": [
        "hidden gem not many tourists visit",
        "unspoiled nature away from the crowds",
        "lesser-known medieval village with local charm",
    ],
}

# Print comparison for every query, grouped by category
for category, queries in MORE_QUERIES.items():
    print(f"\n{'═'*90}")
    print(f"  {category.upper()}")
    print(f"{'═'*90}")
    for q in queries:
        print_comparison(q, k=5)


══════════════════════════════════════════════════════════════════════════════════════════
  SUMMER BEACHES
══════════════════════════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────────────────────────
Query: 'summer beaches'
──────────────────────────────────────────────────────────────────────────────────────────
#    SPARSE (TF-IDF)             DENSE (SentenceTransformer)  HYBRID RRF                  HYBRID Linear α=0.5
──────────────────────────────────────────────────────────────────────────────────────────
1    Dolomites                   French Riviera              French Riviera              French Riviera
2    Stonehenge                  Algarve Beaches             Mont Saint-Michel           Dolomites
3    Eiffel Tower                Mykonos                     Santorini                   Stonehenge
4    Mont Saint-Michel           Santorini                   Ibiza                       

## 11. Recall evaluation — sparse vs dense vs hybrid

Brute-force exact search on the concatenated embedding `[sparse || dense]` is used as ground truth.  
We compare each retriever's Recall@K against this oracle.

In [24]:
# Concatenated embeddings for oracle brute-force (no weighting needed — just for ranking)
# We weight sparse by 0.5 and dense by 0.5 before concatenation so neither dominates by dimension count
SPARSE_WEIGHT = 0.5
DENSE_WEIGHT  = 0.5

combined_embs = np.hstack([
    sparse_embs * SPARSE_WEIGHT,
    dense_embs  * DENSE_WEIGHT,
])
# Re-normalise the concatenated vector to unit length
combined_embs = normalize(combined_embs, norm='l2')


def oracle_top_k(query_text: str, k: int) -> set:
    """Exact brute-force over concatenated embedding space."""
    q_sparse = embed_sparse(query_text) * SPARSE_WEIGHT
    q_dense  = embed_dense(query_text)  * DENSE_WEIGHT
    q_combined = np.concatenate([q_sparse, q_dense])
    q_combined = q_combined / (np.linalg.norm(q_combined) + 1e-9)
    sims       = combined_embs @ q_combined
    return set(np.argsort(sims)[::-1][:k])


def recall(results: list, true_ids: set, k: int) -> float:
    """Fraction of true top-k found in results[:k]."""
    result_names = {r[1]['name'] for r in results[:k]}
    true_names   = {PLACES[i]['name'] for i in true_ids}
    return len(result_names & true_names) / k


# Evaluate on all place embeddings as test queries
test_queries = [DOCS[i] for i in range(N)]

print(f"Evaluating {N} test queries...\n")
print(f"{'Method':<22}  {'k=1':>8}  {'k=3':>8}  {'k=5':>8}  {'k=10':>8}")
print("-" * 58)

sparse_index.ef_search = 50
dense_index.ef_search  = 50

for method_name, search_fn in [
    ("Sparse (TF-IDF)",   lambda q, k: sparse_index.search(embed_sparse(q), k=k)),
    ("Dense (SentTrans)", lambda q, k: dense_index.search(embed_dense(q),  k=k)),
    ("Hybrid RRF",        lambda q, k: hybrid_rrf(q,    k=k, fetch=max(20, k*4))),
    ("Hybrid Linear 0.5", lambda q, k: hybrid_linear(q, k=k, alpha=0.5, fetch=max(20, k*4))),
]:
    row = []
    for k in [1, 3, 5, 10]:
        total = 0.0
        for q in test_queries:
            true_ids = oracle_top_k(q, k)
            results  = search_fn(q, k)
            total   += recall(results, true_ids, k)
        row.append(total / len(test_queries))
    print(f"{method_name:<22}  {row[0]:>8.3f}  {row[1]:>8.3f}  {row[2]:>8.3f}  {row[3]:>8.3f}")

Evaluating 47 test queries...

Method                       k=1       k=3       k=5      k=10
----------------------------------------------------------
Sparse (TF-IDF)            0.660     0.454     0.404     0.366
Dense (SentTrans)          0.681     0.518     0.540     0.504
Hybrid RRF                 0.468     0.340     0.345     0.462
Hybrid Linear 0.5          0.489     0.560     0.523     0.540


## 12. Alpha sweep — find the best fusion weight

How much should we trust dense vs sparse?  
Sweep α from 0 (pure sparse) to 1 (pure dense) and measure Recall@5.

In [25]:
print(f"{'α (dense weight)':<20}  {'Recall@5':>10}")
print("-" * 35)

best_alpha, best_recall = 0.0, 0.0

for alpha in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    total = 0.0
    for q in test_queries:
        true_ids = oracle_top_k(q, k=5)
        results  = hybrid_linear(q, k=5, alpha=alpha, fetch=20)
        total   += recall(results, true_ids, k=5)
    r5 = total / len(test_queries)
    marker = "  ← best" if r5 > best_recall else ""
    if r5 > best_recall:
        best_alpha, best_recall = alpha, r5
    print(f"  α = {alpha:<14.1f}  {r5:>10.3f}{marker}")

print(f"\nBest α = {best_alpha}  →  Recall@5 = {best_recall:.3f}")

α (dense weight)        Recall@5
-----------------------------------
  α = 0.0                  0.404  ← best
  α = 0.1                  0.481  ← best
  α = 0.2                  0.511  ← best
  α = 0.3                  0.511  ← best
  α = 0.4                  0.515  ← best
  α = 0.5                  0.523  ← best
  α = 0.6                  0.519
  α = 0.7                  0.511
  α = 0.8                  0.523
  α = 0.9                  0.532  ← best
  α = 1.0                  0.540  ← best

Best α = 1.0  →  Recall@5 = 0.540


## 13. Incremental insertion into both indexes

New places can be added at any time — just embed with both methods and insert into both indexes.

In [26]:
new_place = {
    "name": "Kotor Bay",
    "country": "Montenegro",
    "description": "Medieval walled city on a fjord-like bay in the Adriatic. "
                   "Byzantine churches, Venetian architecture and dramatic karst mountains.",
    "best_time": "May–October",
    "category": "heritage"
}

new_doc = (
    f"{new_place['name']} {new_place['country']} "
    f"{new_place['category']} {new_place['description']}"
)

# Embed with both methods
new_sparse_vec = embed_sparse(new_doc)
new_dense_vec  = embed_dense(new_doc)

# Insert into both indexes — no rebuilding required
sid = sparse_index.insert(new_sparse_vec, label=new_place)
did = dense_index.insert(new_dense_vec,   label=new_place)

print(f"Inserted '{new_place['name']}' → sparse node {sid}, dense node {did}")
print(f"Sparse index now: {len(sparse_index.vectors)} nodes")
print(f"Dense index now : {len(dense_index.vectors)} nodes")

# Hybrid search for the new place
print(f"\nHybrid RRF results for '{new_place['name']}':")
print(f"{'Rank':<5} {'Score':>8}  {'Place':<28} {'Country':<15} Category")
print("-" * 72)
for rank, (score, place) in enumerate(hybrid_rrf(new_doc, k=5), 1):
    print(f"  {rank:<4} {score:>8.4f}  {place['name']:<28} {place['country']:<15} {place['category']}")

Inserted 'Kotor Bay' → sparse node 48, dense node 48
Sparse index now: 49 nodes
Dense index now : 49 nodes

Hybrid RRF results for 'Kotor Bay':
Rank     Score  Place                        Country         Category
------------------------------------------------------------------------
  1      0.0650  Kotor Bay                    Montenegro      heritage
  2      0.0300  Stockholm Gamla Stan         Sweden          heritage
  3      0.0296  Meteora                      Greece          heritage
  4      0.0290  Acropolis of Athens          Greece          heritage
  5      0.0269  Keukenhof Gardens            Netherlands     nature


## Summary

| | Sparse (TF-IDF) | Dense (SentTrans) | Hybrid RRF | Hybrid Linear |
|--|--|--|--|--|
| **Exact keyword match** | ✅ Best | ❌ Misses synonyms | ✅ Inherited | ✅ Inherited |
| **Semantic / paraphrase** | ❌ Misses | ✅ Best | ✅ Inherited | ✅ Inherited |
| **Score normalisation needed** | — | — | ❌ No (rank-based) | ✅ Yes (min-max) |
| **Tunable weight** | — | — | RRF-k (robust default=60) | α (tune per dataset) |
| **Incremental insert** | ✅ | ✅ | ✅ | ✅ |

**When to use each fusion method:**
- **RRF** — default choice; no score calibration, robust across domains
- **Linear α** — when you have a validation set to tune α; can squeeze out extra recall